# Dependencias

Abra o cmd e execute o comando abaixo:

Reinicie sua IDE após a instalação.

In [ ]:
%%cmd
winget install Gyan.FFmpeg

Microsoft Windows [vers�o 10.0.26200.8037]
(c) Microsoft Corporation. Todos os direitos reservados.

(AI-Audio-Transcription-Examples) c:\Users\lucasmg-cogna\Dev\AI-Audio-Transcription-Examples\src>ffmpeg -version


ffmpeg version 8.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
built with gcc 15.2.0 (Rev11, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-liboapv --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxavs2 --enable-libxeve --enable-libxvid --enable-libaom --enable-libjxl --enable

# Execução do código

In [ ]:
# usado para criar barras de progesso em loops e iterações.
from tqdm.notebook import tqdm 

# python princial lib usada para machine learning, criação e desenvolvimento de modelos de IA. 
import torch 

# lib criada pelo hugging face para manipulação de inferencia e treinamento de modelos.
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor 

# lib usada para analise e processamento de audio.
import librosa 

c:\Users\lucasmg-cogna\Dev\AI-Audio-Transcription-Examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# definido dispositivo que vamos usar para fazer a inferencia, caso tenha GPU usamos "cuda", se não faz fallback para CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# modelos open source disponibilizados pela openai, modelos mais robusto e otimizado ate o momento.
model_id = "openai/whisper-large-v3-turbo" 

# padrão de bit usados pelo modelos, temos float32, float16, bfloat16: principais mecanimos de otimização de bit para modelos, diminui consumo de ram ou vram, e custo computacional
dtype = torch.bfloat16 

# preprocessador para modelos de audio, abstração criada pelo hf
processor = AutoProcessor.from_pretrained(model_id) 

# carregamento do modelos, abstração criada pelo hf
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id).to(device, dtype=dtype) 

Loading weights: 100%|██████████| 587/587 [00:00<00:00, 3782.60it/s]


In [ ]:
# carregando audio para um formato amigavel e convertendo a taxa de amostragem em 16000Hz/16kHz
audio, sr = librosa.load(r"C:\Users\lucasmg-cogna\Dev\AI-Audio-Transcription-Examples\audio\combined.wav", sr=16000)

# geração de tokens, transformando em dados que a llm consegue entender (transformando em sua linguagem). processo de codificação dos dados.
inputs = processor(audio, return_tensors="pt", sampling_rate=16000).to(device, dtype=dtype) 

In [ ]:
# executando modelos, realizando inferencia, passando tokens.
output = model.generate(
    **inputs,
    language="pt",
    task="transcribe",
    return_timestamps=True,
    return_segments=True,
    return_dict_in_generate=True
)

# o modelos devolve tokens trancritos, precisamos transformar em dados do tipo texto. processo de decodificação dos dados.
text = processor.batch_decode(output["sequences"], skip_special_tokens=True)

# loop para exibir seguimentos das transcrições.
for segment in output["segments"][0]:
    start = segment["start"].item()
    end = segment["end"].item()
    tokens = segment["tokens"]
    text_seg = processor.decode(tokens, skip_special_tokens=True)
    
    
    print(f"start: {start:.2f}s")
    print(f"end: {end:.2f}s")
    print(f"duration: {end - start:.2f}s")
    print(f"text: {text_seg}")
    print(f"tokens: {tokens.tolist()}")
    print()


[' Eu não sabia nada de lá, porque nunca tinha... Veio uma intimação, foi a julgamento. Julgamento, assim, numa sala com o juiz, tudo. Porque ela saiu desse... Foi por isso que nós saímos de perdizes, porque minha mãe trabalhava no emprego. Mas eu odiava muito mal. Fiquei muito mal, só chorava, só chorava, queria morrer.']
start: 0.00s
end: 2.40s
duration: 2.40s
text:  Eu não sabia nada de lá, porque nunca tinha...
tokens: [50365, 2186, 2431, 36388, 8096, 368, 7453, 11, 4021, 13768, 13574, 485, 50485]

start: 2.40s
end: 4.54s
duration: 2.14s
text:  Veio uma intimação, foi a julgamento.
tokens: [50485, 9706, 1004, 2772, 560, 4775, 2917, 11, 6901, 257, 30764, 70, 8824, 13, 50592]

start: 4.54s
end: 7.44s
duration: 2.90s
text:  Julgamento, assim, numa sala com o juiz, tudo.
tokens: [50592, 7174, 70, 8824, 11, 8249, 11, 29080, 37596, 395, 277, 3649, 590, 11, 9379, 13, 50737]

start: 7.44s
end: 9.10s
duration: 1.66s
text:  Porque ela saiu desse...
tokens: [50737, 11287, 7175, 601, 5951, 178